In [1]:
import tensorflow as tf
import numpy as np
import os
import time


In [2]:
text = open('khayyam.txt', 'rb').read().decode(encoding='utf-8')


In [3]:
vocabolaries = sorted(set(text))


In [4]:
vocabolaries

['\n',
 '\r',
 ' ',
 '!',
 ':',
 '|',
 '«',
 '»',
 '،',
 '؟',
 'ء',
 'آ',
 'ؤ',
 'ئ',
 'ا',
 'ب',
 'ة',
 'ت',
 'ث',
 'ج',
 'ح',
 'خ',
 'د',
 'ذ',
 'ر',
 'ز',
 'س',
 'ش',
 'ص',
 'ض',
 'ط',
 'ظ',
 'ع',
 'غ',
 'ف',
 'ق',
 'ل',
 'م',
 'ن',
 'ه',
 'و',
 'ي',
 'ً',
 'َ',
 'ُ',
 'ِ',
 'ّ',
 'ْ',
 'ٓ',
 'ٔ',
 'ٰ',
 'پ',
 'چ',
 'ژ',
 'ک',
 'گ',
 'ی']

In [5]:
len(vocabolaries)


57

In [6]:
char2index = {u:i for i, u in enumerate(vocabolaries)}
index2char = np.array(vocabolaries)

In [7]:
index2char[1]


np.str_('\r')

In [8]:
text_as_integer = np.array([char2index[c] for c in text])


In [9]:
text_as_integer

array([ 5, 25, 14, ..., 37,  1,  0], shape=(195185,))

In [10]:
char_dataset = tf.data.Dataset.from_tensor_slices(text_as_integer)


In [11]:
for i in char_dataset.take(10):
    print(index2char[i.numpy()])

|
ز
ا
ه
د
ی
 
ر
ا
 


In [12]:
sequences = char_dataset.batch(101, drop_remainder=True)
sequences

<_BatchDataset element_spec=TensorSpec(shape=(101,), dtype=tf.int64, name=None)>

In [13]:
for i in sequences.take(3):
    print('--->', ''.join(index2char[i.numpy()]))

---> |زاهدی را گفت یاری در عمل
|کم گری تا چشم را ناید خلل
|گفت زاهد از دو بیرون نیست حال
|چشم بیند یا ن
---> بیند آن جمال
|گر ببیند نور حق خود چه غمست
|در وصال حق دو دیده چه کمست
|ور نخواهد دید حق را گو برو
---> 
|این چنین چشم شقی گو کور شو
|غم مخور از دیده، کان عیسی تراست
|چپ مرو تا بخشدت دو چشم راست
|عیسی ر


In [14]:
def sit(batch):
    input_text = batch[:-1]
    target_text = batch[1:]
    return input_text, target_text
dataset = sequences.map(sit)

In [15]:
dataset
for i in dataset.take(1):
    print(''.join(index2char[i[0].numpy()]))
    print(''.join(index2char[i[1].numpy()]))

|زاهدی را گفت یاری در عمل
|کم گری تا چشم را ناید خلل
|گفت زاهد از دو بیرون نیست حال
|چشم بیند یا 
زاهدی را گفت یاری در عمل
|کم گری تا چشم را ناید خلل
|گفت زاهد از دو بیرون نیست حال
|چشم بیند یا ن


In [16]:
dataset = dataset.batch(64, drop_remainder=True)
dataset

<_BatchDataset element_spec=(TensorSpec(shape=(64, 100), dtype=tf.int64, name=None), TensorSpec(shape=(64, 100), dtype=tf.int64, name=None))>

In [17]:
len(vocabolaries)


57

In [18]:
vocabolary_size = len(vocabolaries)
embedding_dim = 256
rnn_units = 1024

In [19]:
model = tf.keras.Sequential([
    tf.keras.layers.Embedding(vocabolary_size, embedding_dim),
    tf.keras.layers.GRU(rnn_units, return_sequences=True),
    tf.keras.layers.Dense(vocabolary_size)
])

In [20]:
for input_text, target_text in dataset.take(1):
    output = model.predict(input_text)
    print(output[0])

2/2 ━━━━━━━━━━━━━━━━━━━━ 3s 267ms/step
[[ 0.00155455 -0.00783508  0.00747598 ... -0.01756554  0.00212991
  -0.00760237]
 [-0.00875618  0.00015468 -0.00612859 ... -0.00776892 -0.00383458
  -0.00165757]
 [ 0.00020277 -0.01323397 -0.00225227 ... -0.00990359  0.00258778
  -0.00562765]
 ...
 [-0.01765322 -0.00552176  0.00890922 ... -0.00163966 -0.00538589
   0.00356533]
 [-0.00514326 -0.0169856   0.00632102 ... -0.00923945  0.00051045
  -0.0032767 ]
 [-0.01535699 -0.00353161  0.00871488 ...  0.00175938 -0.00319223
   0.00341403]]


In [21]:
si = tf.random.categorical(output[0], num_samples=1)
si

<tf.Tensor: shape=(100, 1), dtype=int64, numpy=
array([[37],
       [34],
       [56],
       [21],
       [13],
       [21],
       [41],
       [51],
       [19],
       [41],
       [37],
       [20],
       [54],
       [ 9],
       [12],
       [27],
       [15],
       [22],
       [33],
       [22],
       [17],
       [28],
       [ 0],
       [35],
       [22],
       [10],
       [22],
       [ 7],
       [20],
       [ 6],
       [55],
       [16],
       [53],
       [20],
       [ 9],
       [23],
       [ 2],
       [53],
       [50],
       [ 5],
       [25],
       [21],
       [43],
       [21],
       [ 1],
       [36],
       [20],
       [42],
       [22],
       [ 9],
       [15],
       [49],
       [13],
       [35],
       [25],
       [56],
       [ 2],
       [ 2],
       [50],
       [ 5],
       [33],
       [55],
       [15],
       [27],
       [ 6],
       [39],
       [38],
       [10],
       [14],
       [13],
       [34],
       [53],
       [43],
   

In [22]:
tf.squeeze(si, axis=-1).numpy()


array([37, 34, 56, 21, 13, 21, 41, 51, 19, 41, 37, 20, 54,  9, 12, 27, 15,
       22, 33, 22, 17, 28,  0, 35, 22, 10, 22,  7, 20,  6, 55, 16, 53, 20,
        9, 23,  2, 53, 50,  5, 25, 21, 43, 21,  1, 36, 20, 42, 22,  9, 15,
       49, 13, 35, 25, 56,  2,  2, 50,  5, 33, 55, 15, 27,  6, 39, 38, 10,
       14, 13, 34, 53, 43, 25,  4, 51, 42, 43, 42, 54,  4, 35, 21, 20, 29,
       50, 32, 30,  1, 49, 45, 40, 36, 17, 14,  0, 44, 49, 42,  4])

In [23]:
''.join(index2char[tf.squeeze(si, axis=-1).numpy()])


'مفیخئخيپجيمحک؟ؤشبدغدتص\nقدءد»ح«گةژح؟ذ ژٰ|زخَخ\rلحًد؟بٔئقزی  ٰ|غگبش«هنءائفژَز:پًًَک:قخحضٰعط\rِٔولتا\nًُٔ:'

In [24]:
output[0][0]


array([ 1.5545548e-03, -7.8350808e-03,  7.4759759e-03, -7.1449443e-03,
        9.7297365e-05, -2.0630644e-03, -6.0138572e-03, -1.1118300e-02,
        8.8682603e-03, -6.1041629e-03, -4.5213327e-03,  7.0058610e-03,
        1.0199393e-02, -1.2407557e-02, -8.4847771e-04,  7.2487392e-03,
       -1.2485913e-04, -5.6639579e-03, -6.9267000e-03,  4.0958570e-03,
        1.6297024e-02,  4.1925460e-03,  1.6575206e-02,  1.0534481e-02,
       -2.1386098e-03,  8.2915248e-03,  4.4758366e-03, -6.1044204e-03,
        3.1740745e-03, -7.4997880e-03, -1.9279593e-03, -7.1805203e-03,
        1.9272260e-02,  2.6636403e-03,  8.0898367e-03,  2.1685008e-04,
        3.6324761e-03, -8.0596544e-03, -1.2883528e-02, -5.1537934e-03,
        3.8105207e-03,  2.4311268e-03, -7.2524794e-03, -5.0981934e-03,
        2.5923639e-03, -8.1745796e-03, -4.8817289e-03, -3.7851429e-03,
       -1.5522558e-02, -8.8103791e-04,  1.7524982e-02, -2.7503287e-03,
        1.2205788e-03, -8.8417990e-04, -1.7565541e-02,  2.1299110e-03,
      

In [25]:
model.summary()


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (32, 100, 256)         │        14,592 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru (GRU)                       │ (32, 100, 1024)        │     3,938,304 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (32, 100, 57)          │        58,425 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,011,321 (15.30 MB)

 Trainable params: 4,011,321 (15.30 MB)

 Non-trainable params: 0 (0.00 B)

In [26]:
def loss_f(labels, logits):
    return tf.keras.losses.sparse_categorical_crossentropy(labels, logits, from_logits=True)
model.compile(optimizer='adam', loss=loss_f)

In [27]:
checkpoint = tf.keras.callbacks.ModelCheckpoint(
    filepath='checkpoints/khayyammolana.weights.h5',
    save_weights_only=True,
    save_best_only=True
)

In [65]:
history = model.fit(dataset, epochs=100, callbacks=[checkpoint])


Epoch 1/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 15s 460ms/step - loss: 3.3775
Epoch 2/100


C:\Users\LOQ\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\callbacks\model_checkpoint.py:276: UserWarning: Can save best model only with val_loss available.
  if self._should_save_model(epoch, batch, logs, filepath):


30/30 ━━━━━━━━━━━━━━━━━━━━ 13s 434ms/step - loss: 2.4577
Epoch 3/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 14s 463ms/step - loss: 2.3273
Epoch 4/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 13s 448ms/step - loss: 2.2538
Epoch 5/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 15s 486ms/step - loss: 2.1949
Epoch 6/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 15s 486ms/step - loss: 2.1414
Epoch 7/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 14s 483ms/step - loss: 2.0976
Epoch 8/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 14s 479ms/step - loss: 2.0538
Epoch 9/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 42s 1s/step - loss: 2.0122
Epoch 10/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 56s 539ms/step - loss: 1.9741
Epoch 11/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 14s 469ms/step - loss: 1.9360
Epoch 12/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 14s 465ms/step - loss: 1.8979
Epoch 13/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 15s 497ms/step - loss: 1.8592
Epoch 14/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 15s 509ms/step - loss: 1.8208
Epoch 15/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 15s 502ms/step - loss: 1.7829
Epoch 16/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 15s

In [86]:
model_2 = tf.keras.Sequential([
    tf.keras.layers.Embedding(vocabolary_size, embedding_dim),
    tf.keras.layers.GRU(rnn_units, return_sequences=True),
    tf.keras.layers.Dense(vocabolary_size)
])

In [87]:
model_2.build(input_shape=(None, None))

In [88]:
weights_path = 'checkpoints/khayyammolana.weights.h5'

In [89]:
if os.path.exists(weights_path):
    print(f"فایل پیدا شد: {weights_path}")
    print(f"حجم فایل: {os.path.getsize(weights_path) / (1024*1024):.2f} MB")

فایل پیدا شد: checkpoints/khayyammolana.weights.h5
حجم فایل: 45.93 MB


In [90]:
model_2.load_weights(weights_path)

In [91]:
num_genrate=1000
frist_string='به نام خداوند جان و خرد'
input_eval=[char2index[s]for s in frist_string]
input_eval=tf.expand_dims(input_eval,0)

In [ ]:
text_generate=[]
for i in range(500):
    predic=model_2.predict(input_eval)
    predic=tf.squeeze(predic,axis=0)
    # predic_ids=tf.random.categorical(predic, num_samples=1).numpy()
    predic_ids=np.array(predic.numpy()).argmax(axis=1).reshape(-1,1)[-1][0]
    print(predic_ids)
    massage=np.append(input_eval[0].numpy(),predic_ids)[1:]
    input_eval=tf.expand_dims(massage,0)
    text_generate.append(index2char[predic_ids])

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 117ms/step
8
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 121ms/step
2
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
19
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
14
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
38
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
2
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
15
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
24
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
2
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
34
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
24
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
40
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
21
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
17
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
1
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
0
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
37
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
27
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
17
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
24
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
56
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
2
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
32
1/1 ━━━━━━━━━━━━━━━━━━

In [81]:
''.join(text_generate).split('\n')

['، جان بر فروخت\r',
 '|مشتری علم تحقیقی حقست\r',
 '|دایما بازار او با رونقست\r',
 '|لب ببسته مست در گردون روان\r',
 '|در چمین خوش پنهان گشته اندر زیر پوست\r',
 '|چون نیابد صورت آید در جهان\r',
 '|گرچه جمله زنده اند از بحر جان\r',
 '|آن یکی گفتا بده آن آه را\r',
 '|من دو کوری دارم ای اهل زمان\r',
 '|جسم تو جان گردد و جانت\r',
 '|آن جهاد ظاهر و باطن خدا\r',
 '|تا میان قهر و لطف آن خشم و جن\r',
 '|از له شه می رود از وی منار\r',
 '|وصل نور نی کرد من طافد و خشم\r',
 '|می نماید پیش چشمت کُه بِر همه گر\r',
 '|از سلیمان چون سلیمان روان\r',
 '|من گریزان از تو مانند خیال\r',
 '|کی بیاید بل']

In [85]:
for i in ''.join(text_generate).split('\n'):
    print((i))

، جان بر فروخت
|مشتری علم تحقیقی حقست
|دایما بازار او با رونقست
|لب ببسته مست در گردون روان
|در چمین خوش پنهان گشته اندر زیر پوست
|چون نیابد صورت آید در جهان
|گرچه جمله زنده اند از بحر جان
|آن یکی گفتا بده آن آه را
|من دو کوری دارم ای اهل زمان
|جسم تو جان گردد و جانت
|آن جهاد ظاهر و باطن خدا
|تا میان قهر و لطف آن خشم و جن
|از له شه می رود از وی منار
|وصل نور نی کرد من طافد و خشم
|می نماید پیش چشمت کُه بِر همه گر
|از سلیمان چون سلیمان روان
|من گریزان از تو مانند خیال
|کی بیاید بل
